In [2]:
%pip install duckdb pandas pyarrow

   ---------------------------------------- 0.0/15.4 MB ? eta -:--:--
   ---------- ----------------------------- 3.9/15.4 MB 20.4 MB/s eta 0:00:01
   -------------------- ------------------- 7.9/15.4 MB 20.1 MB/s eta 0:00:01
   -------------------------- ------------- 10.2/15.4 MB 16.6 MB/s eta 0:00:01
   ---------------------------------- ----- 13.1/15.4 MB 15.9 MB/s eta 0:00:01
   ---------------------------------------- 15.4/15.4 MB 15.0 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
from pathlib import Path
import time
import duckdb
import numpy as np
import pandas as pd

DATA = Path("C:/Users/Yuktha/Desktop/old course files/computational tools for data analysis sem 2/books_50pct")   
REVIEWS = DATA / "books_50pct_users_reviews.parquet"
META = DATA / "books_50pct_users_meta.parquet"
CLEAN_META = DATA / "books_clean_text_metadata.parquet"   
SBERT = DATA / "books_sbert_embeddings.npy"
TFIDF = DATA / "books_tfidf_svd_embeddings.npy"
DUPES = DATA / "book_duplicate_clusters.parquet"
COMMUNITIES = DATA / "communities.parquet"                

# Settings 
CUTOFF = "2014-01-01"       # date cut off for cold start; books published after this date are considered new, and books published before this date are considered established.
MIN_NEW_READERS = 3         # a new book needs this many readers after the cut-off
MIN_OLD_REVIEWS = 5         # an established book needs this many reviews before the cut-off
MAX_READER_REVIEWS = 500    # readers above this are left out of the answer key
K = 10                      # recommendations per book
SEED = 42

for p in [REVIEWS, META, CLEAN_META, SBERT, TFIDF, DUPES]:
    print("OK     " if p.exists() else "MISSING", p)

con = duckdb.connect(str(DATA / "coldstart.duckdb"))

OK      C:\Users\Yuktha\Desktop\old course files\computational tools for data analysis sem 2\books_50pct\books_50pct_users_reviews.parquet
OK      C:\Users\Yuktha\Desktop\old course files\computational tools for data analysis sem 2\books_50pct\books_50pct_users_meta.parquet
OK      C:\Users\Yuktha\Desktop\old course files\computational tools for data analysis sem 2\books_50pct\books_clean_text_metadata.parquet
OK      C:\Users\Yuktha\Desktop\old course files\computational tools for data analysis sem 2\books_50pct\books_sbert_embeddings.npy
OK      C:\Users\Yuktha\Desktop\old course files\computational tools for data analysis sem 2\books_50pct\books_tfidf_svd_embeddings.npy
OK      C:\Users\Yuktha\Desktop\old course files\computational tools for data analysis sem 2\books_50pct\book_duplicate_clusters.parquet


In [9]:
# Loading reviews 
con.execute(f"""
    CREATE OR REPLACE TABLE raw_reviews AS
    SELECT user_id, item_id, rating, make_timestamp(timestamp * 1000000) AS ts  -- seconds -> date
    FROM '{REVIEWS.as_posix()}' """)

con.sql("""
    SELECT count(*) AS reviews, count(DISTINCT user_id) AS users, count(DISTINCT item_id) AS books,
           min(ts) AS first, max(ts) AS last
    FROM raw_reviews""").df()

,reviews,users,books,first,last
0,4408904,301553,349741,1996-05-20,2014-07-23


In [10]:
# Flagging book metadata 
meta = pd.read_parquet(META, columns=["item_id", "title", "description", "categories"])
meta["item_id"] = meta["item_id"].astype(str)

def first_genre(cats):
    """[["Books", "Mystery, Thriller & Suspense", ...], ...] -> "Mystery, Thriller & Suspense", or None for plain [["Books"]]."""
    try:
        for path in cats:
            path = list(path)
            if len(path) > 1:
                return str(path[1])
    except TypeError:
        pass
    return None

def text_len(s):
    return s.fillna("").astype(str).str.strip().str.len()

meta["genre"] = meta["categories"].apply(first_genre)
meta["desc_len"] = text_len(meta["description"])
meta["has_description"] = meta["desc_len"] > 0
meta["has_text"] = meta["has_description"] | (text_len(meta["title"]) > 0)
print(f"Books: {len(meta):,} | description: {meta['has_description'].mean():.1%} | "
      f"title or description: {meta['has_text'].mean():.1%} | genre: {meta['genre'].notna().mean():.1%}")

Books: 349,741 | description: 64.9% | title or description: 83.3% | genre: 22.1%


In [11]:
# Merging duplicate editions 
SIMILARITY_THRESHOLD = 0.9

def normalise(s):
    return s.fillna("").astype(str).str.lower().str.replace(r"[^a-z0-9 ]", "", regex=True).str.strip()

dupes = pd.DataFrame({"item_id": pd.Series(dtype=str), "canonical_item_id": pd.Series(dtype=str)})
if DUPES.exists() and SBERT.exists():
    d = pd.read_parquet(DUPES)[["item_id", "canonical_item_id"]].astype(str).drop_duplicates("item_id")
    d = d[d["item_id"] != d["canonical_item_id"]]
    clean = pd.read_parquet(CLEAN_META, columns=["item_id", "title"])
    clean["item_id"] = clean["item_id"].astype(str)
    title = pd.Series(normalise(clean["title"]).values, index=clean["item_id"])
    row = pd.Series(np.arange(len(clean)), index=clean["item_id"])
    emb = np.load(SBERT, mmap_mode="r")
    a = np.asarray(emb[row[d["item_id"]].to_numpy()], dtype="float32")
    b = np.asarray(emb[row[d["canonical_item_id"]].to_numpy()], dtype="float32")
    cos = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-12)
    same_title = (d["item_id"].map(title) == d["canonical_item_id"].map(title)).to_numpy()
    keep = same_title & (cos >= SIMILARITY_THRESHOLD)
    dupes = d[keep]
    print(f"Duplicate pairs from LSH: {len(d):,} | same title: {same_title.sum():,} | "
          f"same title AND similar text (kept): {keep.sum():,}")
else:
    print("Duplicate file or SBERT embeddings missing: continuing WITHOUT merging editions.")

con.register("dupes", dupes)
con.execute("""
    CREATE OR REPLACE TABLE reviews AS
    SELECT user_id, item_id, rating, ts FROM (
        SELECT r.user_id, coalesce(d.canonical_item_id, r.item_id) AS item_id, r.rating, r.ts,
               row_number() OVER (PARTITION BY r.user_id, coalesce(d.canonical_item_id, r.item_id)
                                  ORDER BY r.ts) AS rn
        FROM raw_reviews r LEFT JOIN dupes d USING (item_id)
    ) WHERE rn = 1 """)

before = con.sql("SELECT count(*), count(DISTINCT item_id) FROM raw_reviews").df().iloc[0].tolist()
after = con.sql("SELECT count(*), count(DISTINCT item_id) FROM reviews").df().iloc[0].tolist()
print(f"Reviews: {before[0]:,} -> {after[0]:,}   Books: {before[1]:,} -> {after[1]:,}")

Duplicate pairs from LSH: 10,728 | same title: 7,356 | same title AND similar text (kept): 819
Reviews: 4,408,904 -> 4,408,812   Books: 349,741 -> 348,922


In [12]:
# Date cut-off
con.sql("""
    WITH firsts AS (SELECT item_id, min(ts) AS first FROM reviews GROUP BY item_id),
         dates AS (SELECT unnest([TIMESTAMP '2013-07-01', TIMESTAMP '2014-01-01',
                                  TIMESTAMP '2014-03-01']) AS c)
    SELECT strftime(c, '%Y-%m-%d') AS cutoff,
           (SELECT avg(CASE WHEN ts >= c THEN 1 ELSE 0 END) FROM reviews) AS share_reviews_after,
           (SELECT count(*) FROM firsts WHERE first >= c) AS books_first_after
    FROM dates ORDER BY c """).df()

,cutoff,share_reviews_after,books_first_after
0,2013-07-01,0.391432,60203
1,2014-01-01,0.222346,28820
2,2014-03-01,0.154776,18967


In [13]:
# Splitting books into established vs new
clean_ids = pd.read_parquet(CLEAN_META, columns=["item_id"])["item_id"].astype(str)
book_info = meta[["item_id", "has_description", "has_text", "genre", "desc_len"]].copy()
book_info["has_embedding"] = book_info["item_id"].isin(set(clean_ids))
con.register("book_info", book_info)

con.execute(f"""
    CREATE OR REPLACE TABLE book_stats AS
    SELECT item_id,
           min(ts) AS first_review,
           count(*) FILTER (WHERE ts <  TIMESTAMP '{CUTOFF}') AS n_before,
           count(*) FILTER (WHERE ts >= TIMESTAMP '{CUTOFF}') AS n_after
    FROM reviews GROUP BY item_id """)

con.execute(f"""
    CREATE OR REPLACE TABLE new_books AS
    SELECT s.item_id, s.n_after, b.genre, b.desc_len
    FROM book_stats s JOIN book_info b USING (item_id)
    WHERE s.first_review >= TIMESTAMP '{CUTOFF}'
      AND s.n_after >= {MIN_NEW_READERS}
      AND b.has_description AND b.has_embedding """)

con.execute(f"""
    CREATE OR REPLACE TABLE established AS
    SELECT s.item_id, s.n_before, b.genre,
           coalesce(b.has_embedding AND b.has_text, false) AS usable_for_similarity
    FROM book_stats s LEFT JOIN book_info b USING (item_id)
    WHERE s.first_review < TIMESTAMP '{CUTOFF}' AND s.n_before >= {MIN_OLD_REVIEWS} """)

con.sql("SELECT (SELECT count(*) FROM new_books) AS new_books, (SELECT count(*) FROM established) AS established").df()

,new_books,established
0,9108,172385


In [14]:
# Answer key: established books and their new readers
con.execute(f"""
    CREATE OR REPLACE TABLE answer_key AS
    WITH normal_readers AS (
        SELECT user_id FROM reviews GROUP BY user_id HAVING count(*) <= {MAX_READER_REVIEWS}
    ),
    new_readers AS (
        SELECT DISTINCT r.item_id AS new_id, r.user_id
        FROM reviews r
        JOIN new_books n USING (item_id)
        JOIN normal_readers USING (user_id)
        WHERE r.ts >= TIMESTAMP '{CUTOFF}'
    )
    SELECT nr.new_id, r.item_id AS est_id, count(DISTINCT nr.user_id) AS shared_readers
    FROM new_readers nr
    JOIN reviews r USING (user_id)
    JOIN established e ON r.item_id = e.item_id
    GROUP BY nr.new_id, r.item_id """)

con.sql("SELECT count(DISTINCT new_id) AS new_books_with_answers, count(*) AS pairs, "
        "median(shared_readers) AS median_shared FROM answer_key").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,new_books_with_answers,pairs,median_shared
0,9106,5211359,1.0


In [15]:
# Test split and scoring; Validation
def load_truth(min_shared=1):
    df = con.sql(f"SELECT new_id, est_id FROM answer_key WHERE shared_readers >= {min_shared}").df()
    return df.groupby("new_id")["est_id"].apply(set).to_dict()

truth = load_truth(1)
truth_strict = load_truth(2)

rng = np.random.default_rng(SEED)
all_new = np.array(sorted(truth))
is_val = rng.random(len(all_new)) < 0.5
val_ids, test_ids = set(all_new[is_val]), set(all_new[~is_val])
print(f"Validation books: {len(val_ids):,}   Test books: {len(test_ids):,}")

def evaluate(recs, truth, books, name):
    """recs: dict new_id -> ranked list of established ids. Scores only the given books."""
    hits, precs = [], []
    for b in books:
        if b not in truth:
            continue
        n_correct = len(set(recs.get(b, [])[:K]) & truth[b])
        hits.append(n_correct > 0)
        precs.append(n_correct / K)
    return {"method": name, "hit_rate@10": np.mean(hits), "precision@10": np.mean(precs), "n_books": len(hits)}

Validation books: 4,579   Test books: 4,527


In [16]:
# 0 & 1 Baselines
# 0: the same 10 most-reviewed established books for every new book.
# 1: the 10 most-reviewed books in the new book’s genre
est = con.sql("SELECT item_id, n_before, genre, usable_for_similarity FROM established").df()
pop = est.set_index("item_id")["n_before"]
top_popular = pop.sort_values(ascending=False).index[:K].tolist()
recs_popular = {b: top_popular for b in truth}

genre_top = (est.dropna(subset=["genre"]).sort_values("n_before", ascending=False)
                .groupby("genre")["item_id"].apply(lambda s: s.head(K).tolist()).to_dict())
new_info = con.sql("SELECT item_id, n_after, genre, desc_len FROM new_books").df().set_index("item_id")
recs_genre = {b: genre_top.get(new_info["genre"].get(b), top_popular) for b in truth}

pd.DataFrame([evaluate(recs_popular, truth, test_ids, "0: most popular"),
              evaluate(recs_genre, truth, test_ids, "1: popular in same genre")])

,method,hit_rate@10,precision@10,n_books
0,0: most popular,0.457698,0.111111,4527
1,1: popular in same genre,0.411310,0.099978,4527


In [17]:
# Description similarity: SBERT and TF-IDF
row_of = {item_id: i for i, item_id in enumerate(clean_ids)}
cand_ids = [b for b in est.loc[est["usable_for_similarity"], "item_id"] if b in row_of]
cand_pop = np.log1p(pop.loc[cand_ids].to_numpy())
cand_pop = cand_pop / cand_pop.max()                     # popularity score between 0 and 1
query_ids = [b for b in all_new if b in row_of]
N_CAND = 100                                            # keep the 100 most similar for re-ranking

def load_unit(npy_path):
    emb = np.load(npy_path).astype("float32")
    return emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)

def nearest(emb, chunk=256):
    """For each new book: indices (into cand_ids) and similarities of its N_CAND most similar established books."""
    E = emb[[row_of[b] for b in cand_ids]]
    out = {}
    for start in range(0, len(query_ids), chunk):
        batch = query_ids[start:start + chunk]
        S = emb[[row_of[b] for b in batch]] @ E.T
        top = np.argpartition(-S, N_CAND, axis=1)[:, :N_CAND]
        for r, b in enumerate(batch):
            idx = top[r][np.argsort(-S[r, top[r]])]
            out[b] = (idx, S[r, idx])
    return out

def to_recs(near, lam=0.0):
    """Rank candidates by similarity + lam * popularity; lam = 0 is pure similarity."""
    recs = {}
    for b, (idx, sims) in near.items():
        order = np.argsort(-(sims + lam * cand_pop[idx]))[:K]
        recs[b] = [cand_ids[j] for j in idx[order]]
    return recs

t0 = time.time()
emb_sbert = load_unit(SBERT)
near_sbert = nearest(emb_sbert)
exact_seconds = time.time() - t0
near_tfidf = nearest(load_unit(TFIDF))
recs_sbert, recs_tfidf = to_recs(near_sbert), to_recs(near_tfidf)

pd.DataFrame([evaluate(recs_tfidf, truth, test_ids, "2a: similar (TF-IDF + SVD)"),
              evaluate(recs_sbert, truth, test_ids, "2b: similar (SBERT)")])

,method,hit_rate@10,precision@10,n_books
0,2a: similar (TF-IDF + SVD),0.309035,0.060106,4527
1,2b: similar (SBERT),0.411310,0.089287,4527


In [18]:
# Similarity + popularity
grid = [0, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0]
tuning = pd.DataFrame([{**evaluate(to_recs(near_sbert, lam), truth, val_ids, "val"), "lam": lam} for lam in grid])
display(tuning[["lam", "hit_rate@10", "precision@10"]])
best_lam = tuning.sort_values(["hit_rate@10", "precision@10"], ascending=False)["lam"].iloc[0]
print("Best lam on validation:", best_lam)
recs_rerank = to_recs(near_sbert, best_lam)
pd.DataFrame([evaluate(recs_rerank, truth, test_ids, f"2c: SBERT + popularity (lam={best_lam})")])

,lam,hit_rate@10,precision@10
0,0.00,0.409696,0.090304
1,0.02,0.412972,0.092487
2,0.05,0.421708,0.095960
3,0.10,0.440052,0.102249
4,0.20,0.466041,0.112776
5,0.30,0.473029,0.116794
6,0.50,0.479581,0.119262
7,1.00,0.467133,0.114152


Best lam on validation: 0.5


,method,hit_rate@10,precision@10,n_books
0,2c: SBERT + popularity (lam=0.5),0.481997,0.119748,4527


In [19]:
# LSH
def lsh_index(X, n_bits, n_tables, seed=SEED):
    """Each table: n_bits random hyperplanes; a vector's bucket = which side of each plane it falls on."""
    r = np.random.default_rng(seed)
    planes = r.normal(size=(n_tables, n_bits, X.shape[1])).astype("float32")
    weights = 1 << np.arange(n_bits)
    tables = []
    for t in range(n_tables):
        codes = ((X @ planes[t].T) > 0).astype(np.int64) @ weights
        tables.append(pd.Series(np.arange(len(X))).groupby(codes).apply(np.array).to_dict())
    return planes, weights, tables

def lsh_query(q, planes, weights, tables):
    found = [tables[t].get(int(((planes[t] @ q) > 0).astype(np.int64) @ weights), np.empty(0, int))
             for t in range(len(tables))]
    return np.unique(np.concatenate(found))

def run_lsh(n_bits, n_tables):
    t0 = time.time()
    planes, weights, tables = lsh_index(E, n_bits, n_tables)
    recs, n_compared, overlap = {}, [], []
    for b in query_ids:
        q = emb_sbert[row_of[b]]
        cand = lsh_query(q, planes, weights, tables)
        n_compared.append(len(cand))
        top = cand[np.argsort(-(E[cand] @ q))[:K]] if len(cand) else []
        recs[b] = [cand_ids[j] for j in top]
        overlap.append(len(set(recs[b]) & set(recs_sbert[b])) / K)
    return recs, {"bits": n_bits, "tables": n_tables, "seconds": round(time.time() - t0, 1),
                  "books compared (avg)": round(np.mean(n_compared)),
                  "share of exact top-10 found": round(np.mean(overlap), 3),
                  "hit_rate@10 (test)": round(evaluate(recs, truth, test_ids, "")["hit_rate@10"], 3)}

E = emb_sbert[[row_of[b] for b in cand_ids]]
bits = max(4, int(np.log2(len(E) / 50)))      # aim for roughly 50 books per bucket
lsh_runs = [run_lsh(bits, 10), run_lsh(bits, 30), run_lsh(bits - 2, 30)]
print(f"Exact search: {exact_seconds:.1f}s, compares each new book with all {len(E):,} candidates")
lsh_table = pd.DataFrame([info for _, info in lsh_runs])
display(lsh_table)
recs_lsh = lsh_runs[int(lsh_table["share of exact top-10 found"].idxmax())][0]   # accurate setting

Exact search: 18.5s, compares each new book with all 152,223 candidates


,bits,tables,seconds,books compared (avg),share of exact top-10 found,hit_rate@10 (test)
0,11,10,20.5,2541,0.212,0.300
1,11,30,61.4,8536,0.487,0.368
2,9,30,245.5,23771,0.716,0.395


In [20]:
# Communities (communities.parquet - item_id,community)
# Predicts new book community, 10 most popular books & recommends most similar established books within that community
recs_comm, recs_comm_sim = {}, {}
if COMMUNITIES.exists():
    comm = pd.read_parquet(COMMUNITIES).astype({"item_id": str}).set_index("item_id")["community"]
    est_comm = est.assign(community=est["item_id"].map(comm)).dropna(subset=["community"])
    comm_top = (est_comm.sort_values("n_before", ascending=False)
                        .groupby("community")["item_id"].apply(lambda s: s.head(K).tolist()).to_dict())
    for b, (idx, sims) in near_sbert.items():
        neighbours = [comm.get(cand_ids[j]) for j in idx[:20]]          # 20 most similar books
        neighbours = [c for c in neighbours if pd.notna(c)]
        if not neighbours:
            recs_comm[b] = recs_comm_sim[b] = recs_rerank[b]
            continue
        c = pd.Series(neighbours).mode().iloc[0]                         # predicted community
        recs_comm[b] = comm_top.get(c, top_popular)                      
        inside = [cand_ids[j] for j in idx[np.argsort(-(sims + best_lam * cand_pop[idx]))]
                  if comm.get(cand_ids[j]) == c]
        recs_comm_sim[b] = (inside + [x for x in comm_top.get(c, []) if x not in inside])[:K]   # solution 4
    display(pd.DataFrame([evaluate(recs_comm, truth, test_ids, "3: top books of predicted community"),
                          evaluate(recs_comm_sim, truth, test_ids, "4: similar within predicted community")]))
else:
    print("No communities file yet; skipping solutions 3 and 4.")

No communities file yet; skipping solutions 3 and 4.


In [21]:
# Results 
methods = {"0: most popular": recs_popular, "1: popular in same genre": recs_genre,
           "2a: similar (TF-IDF + SVD)": recs_tfidf, "2b: similar (SBERT)": recs_sbert,
           f"2c: SBERT + popularity (lam={best_lam})": recs_rerank, "2d: SBERT via LSH": recs_lsh}
if recs_comm:
    methods.update({"3: top books of predicted community": recs_comm,
                    "4: similar within predicted community": recs_comm_sim})

normal = pd.DataFrame([evaluate(r, truth, test_ids, m) for m, r in methods.items()]).set_index("method")
strict = pd.DataFrame([evaluate(r, truth_strict, test_ids, m) for m, r in methods.items()]).set_index("method")
summary = normal[["hit_rate@10", "precision@10"]].join(
    strict[["hit_rate@10", "precision@10"]], rsuffix=" (strict)").round(3)
display(summary)

summary.to_csv(DATA / "cold_start_results.csv")
con.sql("SELECT * FROM new_books").df().to_parquet(DATA / "new_books.parquet", index=False)
con.sql("SELECT * FROM answer_key").df().to_parquet(DATA / "answer_key.parquet", index=False)
print("Saved to", DATA)

,hit_rate@10,precision@10,hit_rate@10 (strict),precision@10 (strict)
method,,,,
0: most popular,0.458,0.111,0.192,0.041
1: popular in same genre,0.411,0.100,0.162,0.036
2a: similar (TF-IDF + SVD),0.309,0.060,0.179,0.030
2b: similar (SBERT),0.411,0.089,0.282,0.055
2c: SBERT + popularity (lam=0.5),0.482,0.120,0.325,0.072
2d: SBERT via LSH,0.395,0.082,0.265,0.050


Saved to C:\Users\Yuktha\Desktop\old course files\computational tools for data analysis sem 2\books_50pct


In [23]:
# Did it work better for books with more readers, or with longer descriptions, or in certain genres?
rows = [{"item_id": b, "hit": len(set(recs_rerank.get(b, [])[:K]) & truth[b]) > 0} for b in test_ids]
per_book = pd.DataFrame(rows).set_index("item_id").join(new_info)
per_book["readers"] = pd.cut(per_book["n_after"], [0, 5, 10, 25, np.inf], labels=["3-5", "6-10", "11-25", "26+"])
per_book["description"] = pd.cut(per_book["desc_len"], [0, 200, 600, 1500, np.inf],
                                 labels=["<200 chars", "200-600", "600-1500", "1500+"])
display(per_book.groupby("readers", observed=True)["hit"].agg(["mean", "count"]).rename(columns={"mean": "hit rate"}))
display(per_book.groupby("description", observed=True)["hit"].agg(["mean", "count"]).rename(columns={"mean": "hit rate"}))
display(per_book.fillna({"genre": "(no genre)"}).groupby("genre")["hit"].agg(["mean", "count"])
                .sort_values("count", ascending=False).head(10).rename(columns={"mean": "hit rate"}))

,hit rate,count
readers,,
3-5,0.270011,1774
6-10,0.424088,1179
11-25,0.677652,971
26+,0.903814,603


,hit rate,count
description,,
<200 chars,0.413306,496
200-600,0.440361,1551
600-1500,0.525333,1875
1500+,0.510744,605


,hit rate,count
genre,,
(no genre),0.518123,3090
Literature & Fiction,0.453224,1101
Children's Books,0.254902,51
Christian Books & Bibles,0.317073,41
Gay & Lesbian,0.428571,35
Teen & Young Adult,0.310345,29
Business & Money,0.285714,28
"Health, Fitness & Dieting",0.148148,27
"Cookbooks, Food & Wine",0.153846,26


In [ ]:
# Method comparison within each group (readers & description length)
def hits(recs):
    return pd.Series({b: len(set(recs.get(b, [])[:K]) & truth[b]) > 0 for b in test_ids})

per_book["popular"] = hits(recs_popular)
per_book["sbert_only"] = hits(recs_sbert)
per_book = per_book.rename(columns={"hit": "sbert_plus_pop"}) if "hit" in per_book else per_book
cols = ["sbert_plus_pop", "sbert_only", "popular"]
display(per_book.groupby("readers", observed=True)[cols].mean().round(3))
display(per_book.groupby("description", observed=True)[cols + ["n_after"]]
                .agg({**{c: "mean" for c in cols}, "n_after": "median"}).round(3))

,sbert_plus_pop,sbert_only,popular
readers,,,
3-5,0.270,0.242,0.215
6-10,0.424,0.386,0.416
11-25,0.678,0.546,0.660
26+,0.904,0.743,0.929


,sbert_plus_pop,sbert_only,popular,n_after
description,,,,
<200 chars,0.413,0.333,0.452,7.0
200-600,0.440,0.378,0.384,6.0
600-1500,0.525,0.455,0.510,8.0
1500+,0.511,0.423,0.491,7.0


**Setup.** Cut-off 1 January 2014. 9,108 new books (first reviewed after the cut-off, with a description and at least 3 readers) and 172,385 established books. Answer key: established books also read by each new book's readers (5.2M pairs). New books split 50/50 into validation (tuning) and test (reported scores).

**Results on 4,527 test books (hit rate@10):**

| Method | Normal | Strict (2+ shared readers) |
|---|---|---|
| 0: Most popular | 0.458 | 0.192 |
| 1: Popular in same genre | 0.411 | 0.162 |
| 2a: TF-IDF + SVD similarity | 0.309 | 0.179 |
| 2b: SBERT similarity | 0.411 | 0.282 |
| 2c: SBERT + popularity (λ = 0.5) (!!) | **0.482** | **0.325** |
| 2d: SBERT via LSH (9 bits, 30 tables) | 0.395 | 0.265 |

**Results**
- SBERT + popularity is the best method. Its advantage is largest on the strict test (+69% over popularity), so descriptions capture shared taste rather than general popularity.
- The gain is largest for the least-read new books (3–5 readers: 0.270 vs 0.215, +26%); for widely read new books (26+ readers), popularity wins.
- Descriptions under 200 characters are too short to help (worse than popularity); from 200 characters on, the method beats popularity.
- SBERT clearly beats TF-IDF + SVD. Genre labels exist for only 22% of books and do not beat popularity.
- LSH reduces comparisons 6–60× and keeps up to 72% of the exact top-10, but at this scale exact search (18.5 s) is faster, because it is one matrix multiplication.
- Duplicate check: of 10,728 pairs proposed by title-only MinHash, only 819 had identical titles and very similar text (cosine ≥ 0.9); only these were merged.

Solutions 3 and 4 (Step 12) once the graph file (built from reviews before 2014-01-01) is available.